In [51]:
import copy 
import time 
import torch 
import torch.nn as nn
from torchvision import transforms , datasets ,models
from torch.utils.data import DataLoader,random_split
import torch.optim as optim
import optuna

from sklearn.metrics import (accuracy_score , precision_score , recall_score, f1_score, confusion_matrix , roc_auc_score ,classification_report)



In [55]:
data_dir = "./chest_xray"
n_trials = 15 
tune_epochs = 4 
final_epochs = 20
patience = 5
IMG_SIZE =  224
NUM_CLASSES = 2
seed = 42
OUTPUT_PATH = 'best_model.pt'
BATCH_SIZE = 32
LR = 1e-4
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(DEVICE)

mps


In [36]:
norm= transforms.Normalize(
    mean=[0.485,0.456,0.406],
    std=[0.229,0.224,0.225]
)
train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),
    transforms.ToTensor(),
    norm
])


eval_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    norm
])


In [ ]:
full_train_ds = datasets.ImageFolder(
    root=f"{data_dir}/train",
    transform=train_tf
)
train_size = int(0.9 * len(full_train_ds))
val_size = len(full_train_ds) - train_size
train_ds, val_ds = random_split(
    full_train_ds,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
val_ds.dataset.transform = eval_tf

test_ds = datasets.ImageFolder(
    root=f"{data_dir}/test",
    transform=eval_tf
)

In [56]:
train_loader = DataLoader(
    train_ds,batch_size = BATCH_SIZE , shuffle = True
)
val_loader = DataLoader(
    val_ds , batch_size=BATCH_SIZE ,shuffle = False
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE ,shuffle = False
)
    

In [57]:
model = models.resnet18(weights='IMAGENET1K_V1')
for params in model.parameters():
    params.requires_grad= False
    
model.fc = nn.Linear(
model.fc.in_features , 2
)
model = model.to(DEVICE)

In [58]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters() , lr = LR)

In [59]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        # Move data to device
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Loss
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Statistics
        total_loss += loss.item()

        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = correct / total

    return avg_loss, accuracy


def evaluate(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)

            _, predicted = outputs.max(1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return all_preds, all_labels


In [64]:
print("\nStarting training...")
best_val_acc = 0.0

for epoch in range(1, final_epochs + 1):

   
    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    # Validation
    val_preds, val_labels = evaluate(
        model,
        val_loader
    )

    val_acc = accuracy_score(val_labels, val_preds)

    print(f"Epoch {epoch}/{final_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

        print(f"New best model saved! (Val Acc = {val_acc:.4f})")

    print("-" * 40)



model.load_state_dict(
    torch.load("best_model.pth")
)


Starting training...
Epoch 1/20
Train Loss: 0.3581
Train Accuracy: 0.8496
Validation Accuracy: 0.8736
New best model saved! (Val Acc = 0.8736)
----------------------------------------
Epoch 2/20
Train Loss: 0.2736
Train Accuracy: 0.8965
Validation Accuracy: 0.9061
New best model saved! (Val Acc = 0.9061)
----------------------------------------
Epoch 3/20
Train Loss: 0.2368
Train Accuracy: 0.9152
Validation Accuracy: 0.9215
New best model saved! (Val Acc = 0.9215)
----------------------------------------
Epoch 4/20
Train Loss: 0.2144
Train Accuracy: 0.9201
Validation Accuracy: 0.9253
New best model saved! (Val Acc = 0.9253)
----------------------------------------
Epoch 5/20
Train Loss: 0.2044
Train Accuracy: 0.9235
Validation Accuracy: 0.9291
New best model saved! (Val Acc = 0.9291)
----------------------------------------
Epoch 6/20
Train Loss: 0.1955
Train Accuracy: 0.9250
Validation Accuracy: 0.9368
New best model saved! (Val Acc = 0.9368)
----------------------------------------


<All keys matched successfully>

In [68]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()
all_preds = []
all_labels= []
all_probs = []


with torch.no_grad():
    for images ,labels in test_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        _ ,predicted = outputs.max(1)
        probs = torch.softmax(outputs , dim=1)[:,1]
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    


In [69]:
print("\n===== Test Results =====")

print(f"Accuracy  : {accuracy_score(all_labels, all_preds):.4f}")
print(f"Precision : {precision_score(all_labels, all_preds):.4f}")
print(f"Recall    : {recall_score(all_labels, all_preds):.4f}")
print(f"F1 Score  : {f1_score(all_labels, all_preds):.4f}")
print(f"ROC-AUC   : {roc_auc_score(all_labels, all_probs):.4f}")


===== Test Results =====
Accuracy  : 0.8381
Precision : 0.8017
Recall    : 0.9846
F1 Score  : 0.8838
ROC-AUC   : 0.9486


In [72]:
print(len(val_ds))
print(len((test_ds)))
print(len(train_ds))

522
624
4694
